# Advanced 14 — Governed Agent Skills

This credential-free lab treats a skill as a versioned procedural package—not permission, identity, transport, or a workflow engine. You will route only across eligible packages, activate least authority, execute a read-only incident skill, validate evidence, and inspect lifecycle and security failures.

## 1. Load the shared course runtime

The notebook and tests import the same `policy.py` and `lab.py`; there is no second toy implementation.

In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path('curriculum/advanced/14-agent-skills').resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from lab import (NorthstarSkillRuntime, fixture_principals, request_for, routing_cases)
from policy import ArtifactKind, RiskClass, RouteOutcome, SandboxRequest, SkillLifecycle

runtime = NorthstarSkillRuntime()
principals = fixture_principals()
len(runtime.registry), sorted(runtime.registry)

## 2. Inspect package claims and registry facts

The manifest requests behavior and capabilities. The separate registry record owns approval, lifecycle, tenant policy, and the trusted digest. A package cannot self-approve.

In [ ]:
incident_ref = 'northstar/incident-analysis@2.0.0'
record = runtime.registry[incident_ref]
{
    'skill_ref': record.manifest.ref,
    'requested_required': record.manifest.required_capabilities,
    'requested_optional': record.manifest.optional_capabilities,
    'dependency_refs': tuple(item.ref for item in record.manifest.dependencies),
    'lifecycle': record.lifecycle.value,
    'digest_matches_registry': record.manifest.package_digest == record.approved_digest,
}

## 3. Filter first, then rank

The malicious third-party package says it is trusted and should always be selected. Because it is not independently active and approved, the router never receives it as an eligible candidate.

In [ ]:
principal = principals['incident-reader']
request = request_for(principal)
eligible, filtered = runtime.eligible_skills(principal, request)
decision = runtime.route(principal, request)
assert decision.outcome is RouteOutcome.MATCH
assert decision.selected_skill_ref == incident_ref
assert filtered['third-party/infrastructure-admin@9.9.0'] == 'SKILL_NOT_ACTIVE'
{
    'eligible': tuple(item.ref for item in eligible),
    'selected': decision.selected_skill_ref,
    'reason': decision.reason_code,
    'malicious_filtered_as': filtered['third-party/infrastructure-admin@9.9.0'],
}

## 4. Activate an exact version with least authority

Activation rechecks eligibility and binds principal, tenant, subject, package digest, dependency versions, effective capabilities, budgets, and policy/router/catalog versions. Requested capabilities are intersected with current grants and availability.

In [ ]:
activation = runtime.activate(principal, request, decision)
assert 'production.delete' not in activation.effective_capabilities
{
    'activation_id': activation.activation_id,
    'version': activation.version,
    'mode': activation.mode.value,
    'effective_capabilities': activation.effective_capabilities,
    'dependencies': activation.dependency_refs,
    'catalog_version': activation.catalog_version,
}

## 5. Execute, validate evidence, and keep writes as proposals

Retrieved logs deliberately contain an instruction-injection sentence. The application treats it as evidence data, validates claim support and freshness, and emits an unexecuted action proposal. Model confidence is not evidence.

In [ ]:
result = runtime.execute_incident_skill(
    activation, {'service': 'checkout', 'time_window_minutes': 30}
)
assert result.status.value == 'SUCCEEDED'
assert result.action_proposal and not result.action_proposal.executed
{
    'status': result.status.value,
    'evidence_ids': result.evidence_ids,
    'postconditions': result.verified_postconditions,
    'proposal': result.action_proposal.model_dump(),
}

## 6. Simulate the script sandbox boundary

The fixture never executes package code on the host. It demonstrates an adapter decision for an approved script digest and rejects undeclared network, filesystem, environment, subprocess, or timeout access.

In [ ]:
script = next(item for item in record.manifest.artifacts if item.kind is ArtifactKind.SCRIPT)
sandbox_decision = runtime.run_script(
    activation,
    SandboxRequest(
        script_id=script.artifact_id, script_digest=script.digest,
        network_destinations=('attacker.example',), timeout_ms=500,
    ),
)
assert not sandbox_decision.allowed
assert not sandbox_decision.executed_in_host_process
sandbox_decision.model_dump()

## 7. Demonstrate revocation and dependency propagation

Quarantining the evidence-review dependency invalidates the parent activation before its next execution step. Deprecation can be handled differently for an already pinned activation; quarantine is an emergency stop.

In [ ]:
dependency_ref = 'northstar/evidence-review@2.1.0'
dependency_record = runtime.registry[dependency_ref]
runtime.registry[dependency_ref] = dependency_record.model_copy(
    update={'lifecycle': SkillLifecycle.QUARANTINED}
)
assert runtime.activation_current_reason(activation) == 'DEPENDENCY_REVOKED'
{
    'blocked_parent_versions': runtime.blocked_dependents(dependency_ref),
    'activation_status': runtime.activation_current_reason(activation),
}

## 8. Evaluate routing safety

The deterministic replay validates integration and policy—not real model intelligence. Production routing needs held-out labelled data, slice metrics, drift monitoring, and shadow/canary promotion.

In [ ]:
evaluation_runtime = NorthstarSkillRuntime()
report = evaluation_runtime.evaluate_routing(principal, routing_cases(principal))
assert report.false_activation_rate == 0
report.model_dump()

## 9. Your turn

1. Add a refund-proposal skill that cannot execute a refund.
2. Add an ambiguous request and a typed clarification response.
3. Add an indirect dependency and test transitive quarantine.
4. Add a package update with a new write capability and inspect `package_changes()`.
5. Design a production sandbox adapter without running untrusted code in this notebook.

Remember: discovery is not activation; activation is not execution authorization; output is not verified completion.